In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SQL_Ecommerce") \
    .master("spark://26.37.93.102:7077") \
    .config("spark.executor.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Đọc dữ liệu sạch từ HDFS rồi đăng ký TempView
df = spark.read.parquet("hdfs://master:9000/data/test_cleaned.parquet")
df.createOrReplaceTempView("ecommerce_cleaned")
print("Đã load TempView ecommerce_cleaned\n")

Đã load TempView ecommerce_cleaned



# CÂU 1 – PHÂN TÍCH PHỄU CHUYỂN ĐỔI

In [2]:
print("=" * 60)
print("CÂU 1 – PHỄU CHUYỂN ĐỔI")
print("=" * 60)

spark.sql("""
          WITH funnel AS (
              SELECT
                  ts_month,
                  cat_0,
                  -- Đếm session duy nhất có hành vi cart
                  COUNT(DISTINCT CASE WHEN event_type = 'cart'     THEN user_session END) AS cart_sessions,
                  -- Đếm session duy nhất có hành vi purchase
                  COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_session END) AS purchase_sessions,
                  -- Tổng session bất kể loại hành vi
                  COUNT(DISTINCT user_session) AS total_sessions
              FROM ecommerce_cleaned
              GROUP BY ts_month, cat_0
          )
          SELECT
              ts_month,
              cat_0,
              total_sessions,
              cart_sessions,
              purchase_sessions,
              -- Tỷ lệ thêm vào giỏ / tổng session
              ROUND(cart_sessions     / total_sessions     * 100, 2) AS cart_rate_pct,
              -- Tỷ lệ mua / thêm vào giỏ (điểm nghẽn chính)
              ROUND(purchase_sessions / NULLIF(cart_sessions, 0) * 100, 2) AS purchase_rate_pct
          FROM funnel
          ORDER BY ts_month, cat_0
          """).show(50, truncate=False)


CÂU 1 – PHỄU CHUYỂN ĐỔI
+--------+------------+--------------+-------------+-----------------+-------------+-----------------+
|ts_month|cat_0       |total_sessions|cart_sessions|purchase_sessions|cart_rate_pct|purchase_rate_pct|
+--------+------------+--------------+-------------+-----------------+-------------+-----------------+
|4       |accessories |15988         |10603        |5734             |66.32        |54.08            |
|4       |apparel     |181365        |115646       |70027            |63.76        |60.55            |
|4       |appliances  |352583        |211691       |151803           |60.04        |71.71            |
|4       |auto        |12924         |8351         |4825             |64.62        |57.78            |
|4       |computers   |46264         |28702        |18641            |62.04        |64.95            |
|4       |construction|672094        |377930       |317653           |56.23        |84.05            |
|4       |country_yard|10093         |6702       

# CÂU 2 – TÌM CƠ HỘI BÁN CHÉO

In [3]:
print("=" * 60)
print("CÂU 2 – CƠ HỘI BÁN CHÉO")
print("=" * 60)

spark.sql("""
          WITH purchases AS (
              -- Lọc riêng các sự kiện purchase
              SELECT user_session, product_id
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
          )
          SELECT
              a.product_id AS product_a,
              b.product_id AS product_b,
              COUNT(*) AS co_purchase_count
          FROM purchases a
                   -- Self-join: cùng session, product_a < product_b để tránh lặp cặp đảo
                   JOIN purchases b
                        ON a.user_session = b.user_session
                            AND a.product_id < b.product_id
          GROUP BY a.product_id, b.product_id
          ORDER BY co_purchase_count DESC
              LIMIT 20
          """).show(20, truncate=False)


CÂU 2 – CƠ HỘI BÁN CHÉO
+---------+---------+-----------------+
|product_a|product_b|co_purchase_count|
+---------+---------+-----------------+
|1005100  |1005212  |1164             |
|22800035 |4803879  |576              |
|1004856  |1005100  |483              |
|100068488|100068493|343              |
|1004836  |1005100  |310              |
|1005169  |1005212  |301              |
|100068488|1004836  |293              |
|22800093 |4803879  |288              |
|22800035 |22800093 |288              |
|1004836  |1005212  |283              |
|100068488|1005212  |262              |
|100068488|1005100  |254              |
|1005160  |1005223  |253              |
|100086203|1005100  |250              |
|1005098  |1005100  |216              |
|1005211  |1005212  |208              |
|1002544  |1005115  |205              |
|1005099  |1005100  |203              |
|1004249  |1005115  |197              |
|1005212  |1005225  |191              |
+---------+---------+-----------------+



# CÂU 3 – PHÂN KHÚC KHÁCH HÀNG DỰA TRÊN GIÁ TRỊ CỐT LÕI

In [4]:
print("=" * 60)
print("CÂU 3 – PHÂN KHÚC KHÁCH HÀNG")
print("=" * 60)

spark.sql("""
          WITH user_spend AS (
              -- Tổng chi tiêu của từng khách hàng
              SELECT
                  user_id,
                  SUM(price)             AS monetary,
                  COUNT(DISTINCT user_session) AS purchase_sessions
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY user_id
          ),
               ranked AS (
                   SELECT
                       user_id,
                       monetary,
                       purchase_sessions,
                       -- Xếp hạng từ người chi nhiều nhất xuống ít nhất
                       ROW_NUMBER() OVER (ORDER BY monetary DESC)  AS rnk,
                       COUNT(*)     OVER ()                        AS total_users,
                       SUM(monetary) OVER ()                       AS total_revenue,
            -- Doanh thu tích lũy từ top xuống
                       SUM(monetary) OVER (
                ORDER BY monetary DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )                                           AS cumulative_revenue
                   FROM user_spend
               )
          SELECT
              user_id,
              ROUND(monetary, 2)                                    AS total_spend,
              purchase_sessions,
              rnk,
              total_users,
              ROUND(rnk * 100.0 / total_users, 2)                  AS user_pct,
              ROUND(cumulative_revenue * 100.0 / total_revenue, 2) AS cumulative_revenue_pct,
              -- Đánh dấu có thuộc nhóm top 20% Pareto không
              CASE
                  WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20% (Pareto)'
                  ELSE 'Remaining 80%'
                  END AS pareto_segment
          FROM ranked
          ORDER BY rnk
              LIMIT 50
          """).show(50, truncate=False)

# Tóm tắt Pareto: bao nhiêu user top 20% đóng góp bao nhiêu % doanh thu
spark.sql("""
          WITH user_spend AS (
              SELECT user_id, SUM(price) AS monetary
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY user_id
          ),
               ranked AS (
                   SELECT
                       user_id, monetary,
                       ROW_NUMBER() OVER (ORDER BY monetary DESC) AS rnk,
                       COUNT(*)      OVER ()                      AS total_users,
                       SUM(monetary) OVER ()                      AS total_revenue,
                       SUM(monetary) OVER (
                ORDER BY monetary DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )                                          AS cumulative_revenue
                   FROM user_spend
               )
          SELECT
              CASE WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20%' ELSE 'Bottom 80%' END AS segment,
              COUNT(*)                                          AS user_count,
              ROUND(SUM(monetary), 2)                          AS segment_revenue,
              ROUND(SUM(monetary) * 100.0 / MAX(total_revenue), 2) AS revenue_pct
          FROM ranked
          GROUP BY CASE WHEN rnk * 1.0 / total_users <= 0.2 THEN 'Top 20%' ELSE 'Bottom 80%' END
          ORDER BY revenue_pct DESC
          """).show(truncate=False)


CÂU 3 – PHÂN KHÚC KHÁCH HÀNG
+---------+-----------+-----------------+---+-----------+--------+----------------------+----------------+
|user_id  |total_spend|purchase_sessions|rnk|total_users|user_pct|cumulative_revenue_pct|pareto_segment  |
+---------+-----------+-----------------+---+-----------+--------+----------------------+----------------+
|553446649|122299.14  |140              |1  |482314     |0.00    |0.05                  |Top 20% (Pareto)|
|566213018|119393.24  |98               |2  |482314     |0.00    |0.1                   |Top 20% (Pareto)|
|518514888|114484.63  |71               |3  |482314     |0.00    |0.15                  |Top 20% (Pareto)|
|628167977|112569.45  |23               |4  |482314     |0.00    |0.19                  |Top 20% (Pareto)|
|522016074|103291.35  |78               |5  |482314     |0.00    |0.24                  |Top 20% (Pareto)|
|548782814|100853.54  |41               |6  |482314     |0.00    |0.28                  |Top 20% (Pareto)|
|6350216

# CÂU 4 – PHÂN TÍCH XU HƯỚNG TĂNG TRƯỞNG BẰNG TRUNG BÌNH ĐỘNG

In [5]:
print("=" * 60)
print("CÂU 4 – PHÂN TÍCH XU HƯỚNG TĂNG TRƯỞNG")
print("=" * 60)

spark.sql("""
          WITH daily_revenue AS (
              -- Gom doanh thu theo ngày từ các đơn purchase
              SELECT
                  TO_DATE(event_time) AS event_date,
                  SUM(price)          AS daily_revenue,
                  COUNT(*)            AS order_count
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY TO_DATE(event_time)
          )
          SELECT
              event_date,
              ROUND(daily_revenue, 2) AS daily_revenue,
              order_count,
              -- Trung bình động 3 ngày (lùi 2 ngày + ngày hiện tại)
              -- Phù hợp với dataset 30 ngày: có giá trị ổn định từ ngày thứ 3
              ROUND(AVG(daily_revenue) OVER (
            ORDER BY event_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS ma_3day,
              -- Chênh lệch tuyệt đối so với MA3
              ROUND(daily_revenue - AVG(daily_revenue) OVER (
            ORDER BY event_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS delta_vs_ma,
              -- Flag đột biến: vượt 40% so với MA3 (ngưỡng thấp hơn vì cửa sổ ngắn hơn)
              CASE
                  WHEN daily_revenue > 1.4 * AVG(daily_revenue) OVER (
                ORDER BY event_date
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) THEN 'SPIKE'
                  WHEN daily_revenue < 0.6 * AVG(daily_revenue) OVER (
                ORDER BY event_date
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) THEN 'DROP'
                  ELSE 'Normal'
                  END AS anomaly_flag
          FROM daily_revenue
          ORDER BY event_date
          """).show(50, truncate=False)

CÂU 4 – PHÂN TÍCH XU HƯỚNG TĂNG TRƯỞNG
+----------+-------------+-----------+-------------+-----------+------------+
|event_date|daily_revenue|order_count|ma_3day      |delta_vs_ma|anomaly_flag|
+----------+-------------+-----------+-------------+-----------+------------+
|2020-04-01|8085211.82   |27086      |8085211.82   |0.0        |Normal      |
|2020-04-02|8915154.58   |29452      |8500183.2    |414971.38  |Normal      |
|2020-04-03|8485596.84   |28748      |8495321.08   |-9724.24   |Normal      |
|2020-04-04|7210820.03   |25281      |8203857.15   |-993037.12 |Normal      |
|2020-04-05|6456220.71   |23717      |7384212.53   |-927991.82 |Normal      |
|2020-04-06|6876365.07   |24896      |6847801.94   |28563.13   |Normal      |
|2020-04-07|6597968.33   |23875      |6643518.04   |-45549.7   |Normal      |
|2020-04-08|7096046.63   |25393      |6856793.34   |239253.29  |Normal      |
|2020-04-09|7224806.48   |26204      |6972940.48   |251866.0   |Normal      |
|2020-04-10|8241727.67   

# CÂU 5 – TOP 3 S THƯƠNG HIỆU CHỦ LỰC THEO NGÀNH HÀNG

In [6]:
print("=" * 60)
print("CÂU 5 – TOP 3 THƯƠNG HIỆU THEO NGÀNH HÀNG")
print("=" * 60)

spark.sql("""
    WITH brand_revenue AS (
        -- 1. Tính tổng doanh thu theo ngành hàng và thương hiệu
        SELECT
            cat_0,
            brand,
            SUM(price) AS total_revenue,
            COUNT(*)   AS order_count
        FROM ecommerce_cleaned
        WHERE event_type = 'purchase'
          AND brand != 'unknown'
        GROUP BY cat_0, brand
    ),
    ranked_brands AS (
        -- 2. Xếp hạng doanh thu trong từng ngành
        SELECT
            cat_0,
            brand,
            ROUND(total_revenue, 2) AS total_revenue,
            order_count,
            DENSE_RANK() OVER (
                PARTITION BY cat_0
                ORDER BY total_revenue DESC
            ) AS brand_rank
        FROM brand_revenue
    )
    -- 3. Lọc lấy Top 3
    SELECT *
    FROM ranked_brands
    WHERE brand_rank <= 3
    ORDER BY cat_0, brand_rank
""").show(50, truncate=False)

CÂU 5 – TOP 3 THƯƠNG HIỆU THEO NGÀNH HÀNG
+------------+---------+-------------+-----------+----------+
|cat_0       |brand    |total_revenue|order_count|brand_rank|
+------------+---------+-------------+-----------+----------+
|accessories |huggies  |16901.04     |1182       |1         |
|accessories |sho-me   |16700.3      |102        |2         |
|accessories |neoline  |13711.3      |90         |3         |
|apparel     |sony     |3387310.62   |8068       |1         |
|apparel     |apple    |1160225.52   |2030       |2         |
|apparel     |samsung  |816552.33    |3145       |3         |
|appliances  |samsung  |9081054.01   |24794      |1         |
|appliances  |lg       |7072750.25   |15676      |2         |
|appliances  |xiaomi   |1577230.53   |3670       |3         |
|auto        |lucente  |64048.33     |208        |1         |
|auto        |sokolov  |31486.5      |53         |2         |
|auto        |jkexer   |25354.5      |50         |3         |
|computers   |lucente  |3028

# CÂU 6 –  ĐO LƯỜNG THỜI GIAN RA QUYẾT ĐỊNH VÀ CHU KÌ MUA SẮM

In [7]:
print("=" * 60)
print("CÂU 6 – THỜI GIAN TỪ GIỎ HÀNG → MUA")
print("=" * 60)

spark.sql("""
    WITH cart_events AS (
        -- Lấy 19 ký tự đầu (cắt bỏ chữ UTC) trước khi chuyển thành timestamp
        SELECT user_id, product_id, user_session,
               MIN(UNIX_TIMESTAMP(LEFT(event_time, 19))) AS cart_ts
        FROM ecommerce_cleaned
        WHERE event_type = 'cart'
        GROUP BY user_id, product_id, user_session
    ),
    purchase_events AS (
        -- Tương tự cho sự kiện mua hàng
        SELECT user_id, product_id, user_session,
               MIN(UNIX_TIMESTAMP(LEFT(event_time, 19))) AS purchase_ts
        FROM ecommerce_cleaned
        WHERE event_type = 'purchase'
        GROUP BY user_id, product_id, user_session
    )
    SELECT
        cat_0,
        ROUND(AVG((p.purchase_ts - c.cart_ts) / 60.0), 2) AS avg_decision_minutes,
        ROUND(MIN((p.purchase_ts - c.cart_ts) / 60.0), 2) AS min_decision_minutes,
        ROUND(MAX((p.purchase_ts - c.cart_ts) / 60.0), 2) AS max_decision_minutes,
        COUNT(*)                                          AS sample_count
    FROM cart_events c
    JOIN purchase_events p
        ON  c.user_id      = p.user_id
        AND c.product_id   = p.product_id
        AND c.user_session = p.user_session
        AND p.purchase_ts  > c.cart_ts
    JOIN ecommerce_cleaned e
        ON  e.user_id    = c.user_id
        AND e.product_id = c.product_id
    GROUP BY cat_0
    ORDER BY avg_decision_minutes
""").show(truncate=False)

CÂU 6 – THỜI GIAN TỪ GIỎ HÀNG → MUA
+------------+--------------------+--------------------+--------------------+------------+
|cat_0       |avg_decision_minutes|min_decision_minutes|max_decision_minutes|sample_count|
+------------+--------------------+--------------------+--------------------+------------+
|stationery  |9.98                |9.98                |9.98                |2           |
|medicine    |42.63               |14.17               |61.60               |5           |
|accessories |164.16              |2.35                |1474.47             |53          |
|furniture   |227.85              |0.78                |4304.58             |501         |
|auto        |231.77              |1.75                |1997.68             |57          |
|country_yard|285.20              |2.88                |2033.22             |36          |
|sport       |307.87              |1.02                |12774.17            |741         |
|appliances  |308.87              |0.20               

# CÂU 7 – PHÂN TÍCH GIỎ HÀNG BỊ BỎ RƠI

In [8]:
print("=" * 60)
print("CÂU 7 – GIỎ HÀNG BỊ BỎ RƠI")
print("=" * 60)

spark.sql("""
          WITH cart_only AS (
              -- Session có cart nhưng không có bất kỳ purchase nào
              SELECT user_session
              FROM ecommerce_cleaned
              GROUP BY user_session
              HAVING SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) > 0
                 AND SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) = 0
          )
          SELECT
              e.cat_0,
              e.brand,
              COUNT(*)             AS abandoned_items,
              ROUND(SUM(e.price), 2) AS lost_revenue
          FROM ecommerce_cleaned e
                   -- Chỉ lấy các dòng thuộc session bỏ giỏ, event_type = cart
                   JOIN cart_only c ON e.user_session = c.user_session
          WHERE e.event_type = 'cart'
            AND e.brand != 'unknown'
          GROUP BY e.cat_0, e.brand
          ORDER BY lost_revenue DESC
              LIMIT 30
          """).show(30, truncate=False)


CÂU 7 – GIỎ HÀNG BỊ BỎ RƠI
+------------+---------+---------------+-------------+
|cat_0       |brand    |abandoned_items|lost_revenue |
+------------+---------+---------------+-------------+
|construction|apple    |101027         |8.768659882E7|
|construction|samsung  |263453         |6.557555296E7|
|construction|xiaomi   |73258          |1.57803609E7 |
|appliances  |samsung  |39371          |1.409710491E7|
|appliances  |lg       |24579          |1.100484832E7|
|electronics |acer     |17371          |9899750.7    |
|electronics |apple    |10481          |8900663.28   |
|construction|huawei   |37137          |7503018.8    |
|electronics |lenovo   |13314          |6915790.9    |
|construction|oppo     |33348          |6810713.1    |
|electronics |asus     |11740          |6054079.94   |
|apparel     |sony     |13995          |5980616.23   |
|unknown     |thermomix|3265           |4867690.53   |
|electronics |hp       |9330           |4633902.13   |
|sport       |apple    |22620         

# CÂU 8 – ĐỊNH VỊ GIÁ, PHÂN KHÚC THƯƠNG HIỆU & ĐỘ NHẠY GIÁ

In [9]:
print("=" * 60)
print("CÂU 8 – ĐỊNH VỊ GIÁ & ĐỘ NHẠY GIÁ")
print("=" * 60)

spark.sql("""
          WITH product_stats AS (
              -- Thống kê giá và chuyển đổi theo sản phẩm
              SELECT
                  product_id, cat_0, brand,
                  AVG(price)  AS avg_price,
                  SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
                  COUNT(*)    AS total_events,
                  ROUND(SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
                            / COUNT(*) * 100, 2) AS conversion_rate
              FROM ecommerce_cleaned
              GROUP BY product_id, cat_0, brand
          ),
               industry_avg AS (
                   -- Giá trung bình toàn ngành hàng (dùng CROSS JOIN để broadcast)
                   SELECT cat_0, AVG(avg_price) AS industry_avg_price
                   FROM product_stats
                   GROUP BY cat_0
               ),
               decile_tagged AS (
                   SELECT
                       p.*,
                       i.industry_avg_price,
                       -- Phân rã theo decile giá (1=rẻ nhất, 10=đắt nhất)
                       NTILE(10) OVER (ORDER BY p.avg_price ASC) AS price_decile,
            -- Phân loại định vị so với trung bình ngành
                       CASE
                           WHEN p.avg_price > 1.2 * i.industry_avg_price THEN 'Premium'
                           WHEN p.avg_price < 0.8 * i.industry_avg_price THEN 'Budget'
                           ELSE 'Mid-range'
                           END AS price_position
                   FROM product_stats p
                            JOIN industry_avg i USING (cat_0)
               )
          SELECT
              price_position,
              price_decile,
              COUNT(DISTINCT product_id)          AS product_count,
              ROUND(AVG(avg_price), 2)            AS avg_price,
              ROUND(AVG(conversion_rate), 2)      AS avg_conversion_rate_pct,
              SUM(purchases)                      AS total_purchases
          FROM decile_tagged
          GROUP BY price_position, price_decile
          ORDER BY price_position, price_decile
          """).show(50, truncate=False)


CÂU 8 – ĐỊNH VỊ GIÁ & ĐỘ NHẠY GIÁ
+--------------+------------+-------------+---------+-----------------------+---------------+
|price_position|price_decile|product_count|avg_price|avg_conversion_rate_pct|total_purchases|
+--------------+------------+-------------+---------+-----------------------+---------------+
|Budget        |1           |9540         |4.26     |36.73                  |23267          |
|Budget        |2           |9503         |9.05     |35.84                  |34136          |
|Budget        |3           |9522         |16.3     |32.43                  |43597          |
|Budget        |4           |9528         |27.34    |30.18                  |60250          |
|Budget        |5           |9530         |42.68    |30.95                  |60359          |
|Budget        |6           |9063         |62.8     |31.24                  |57979          |
|Budget        |7           |5816         |88.65    |32.22                  |47846          |
|Budget        |8         

# CÂU 9 – PHÂN TÍCH HÀNH TRÌNH KHÁCH HÀNG & HÀNH VI BẤT THƯỜNG

In [10]:
print("=" * 60)
print("CÂU 9A – DỰ ĐOÁN HÀNH VI KẾ TIẾP")
print("=" * 60)

spark.sql("""
          WITH session_events AS (
              SELECT
                  user_id, user_session, event_type,
                  UNIX_TIMESTAMP(event_time) AS ts,
                  -- Hành động kế tiếp trong cùng session, sắp xếp theo thời gian
                  LEAD(event_type) OVER (
                PARTITION BY user_id, user_session
                ORDER BY UNIX_TIMESTAMP(event_time)
            ) AS next_action
              FROM ecommerce_cleaned
          )
          SELECT
              event_type    AS current_action,
              next_action,
              COUNT(*)      AS transition_count,
              ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY event_type), 2) AS transition_pct
          FROM session_events
          WHERE next_action IS NOT NULL
          GROUP BY event_type, next_action
          ORDER BY event_type, transition_count DESC
          """).show(truncate=False)

print("=" * 60)
print("CÂU 9B – CHUỖI NGÀY TRUY CẬP LIÊN TỤC")
print("=" * 60)

spark.sql("""
          WITH purchase_days AS (
              -- Ngày mua duy nhất của từng user
              SELECT DISTINCT
                  user_id,
                  TO_DATE(event_time) AS purchase_date,
                  -- Số thứ tự ngày mua theo từng user
                  DENSE_RANK() OVER (
                PARTITION BY user_id
                ORDER BY TO_DATE(event_time)
            ) AS day_rank
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
          ),
               islands AS (
                   -- Trừ rank cho date_diff để tạo group_id cho chuỗi liên tục
                   SELECT *,
                          DATE_SUB(purchase_date, day_rank) AS island_id
                   FROM purchase_days
               )
          SELECT
              user_id,
              MIN(purchase_date)  AS streak_start,
              MAX(purchase_date)  AS streak_end,
              COUNT(*)            AS streak_length_days
          FROM islands
          GROUP BY user_id, island_id
          HAVING COUNT(*) >= 3          -- Chỉ lấy chuỗi từ 3 ngày liên tiếp trở lên
          ORDER BY streak_length_days DESC
              LIMIT 20
          """).show(truncate=False)

print("=" * 60)
print("CÂU 9C – PHÁT HIỆN TRUY CẬP BẤT THƯỜNG")
print("=" * 60)

spark.sql("""
          SELECT
              user_session,
              user_id,
              SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS cart_count,
              SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_count,
              COUNT(DISTINCT product_id)                                 AS distinct_products
          FROM ecommerce_cleaned
          GROUP BY user_session, user_id
          -- Cart > 10 nhưng không có purchase nào → nghi ngờ bot cào giá
          HAVING cart_count > 10
             AND purchase_count = 0
          ORDER BY cart_count DESC
              LIMIT 30
          """).show(30, truncate=False)


CÂU 9A – DỰ ĐOÁN HÀNH VI KẾ TIẾP
+--------------+-----------+----------------+--------------+
|current_action|next_action|transition_count|transition_pct|
+--------------+-----------+----------------+--------------+
|cart          |cart       |634805          |91.68         |
|cart          |purchase   |57636           |8.32          |
|purchase      |purchase   |175544          |91.20         |
|purchase      |cart       |16945           |8.80          |
+--------------+-----------+----------------+--------------+

CÂU 9B – CHUỖI NGÀY TRUY CẬP LIÊN TỤC
+---------+------------+----------+------------------+
|user_id  |streak_start|streak_end|streak_length_days|
+---------+------------+----------+------------------+
|514895049|2020-04-01  |2020-04-19|19                |
|575207664|2020-04-01  |2020-04-19|19                |
|635021608|2020-04-01  |2020-04-19|19                |
|629426279|2020-04-01  |2020-04-19|19                |
|514073817|2020-04-01  |2020-04-18|18                |


# CÂU 10 – BẢN ĐỒ NHIỆT VÀ CHÊNH LỆCH DOANH THU THEO GIỜ

In [11]:
print("=" * 60)
print("CÂU 10 – BẢN ĐỒ NHIỆT & CHÊNH LỆCH DOANH THU THEO GIỜ")
print("=" * 60)

spark.sql("""
          WITH hourly_revenue AS (
              -- Doanh thu theo giờ và ngày trong tuần
              SELECT
                  ts_weekday,
                  ts_hour,
                  SUM(price)  AS revenue,
                  COUNT(*)    AS order_count
              FROM ecommerce_cleaned
              WHERE event_type = 'purchase'
              GROUP BY ts_weekday, ts_hour
          )
          SELECT
              ts_weekday,
              ts_hour,
              ROUND(revenue, 2)     AS revenue,
              order_count,
              -- Doanh thu giờ trước trong cùng ngày trong tuần
              ROUND(LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        ), 2) AS prev_hour_revenue,
              -- Chênh lệch tuyệt đối so với giờ trước
              ROUND(revenue - LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        ), 2) AS revenue_delta,
              -- Tăng trưởng phần trăm so với giờ trước
              ROUND((revenue - LAG(revenue) OVER (
            PARTITION BY ts_weekday
            ORDER BY ts_hour
        )) / NULLIF(LAG(revenue) OVER (
                      PARTITION BY ts_weekday
            ORDER BY ts_hour
                                 ), 0) * 100, 2) AS growth_pct
          FROM hourly_revenue
          ORDER BY ts_weekday, ts_hour
          """).show(100, truncate=False)


CÂU 10 – BẢN ĐỒ NHIỆT & CHÊNH LỆCH DOANH THU THEO GIỜ
+----------+-------+----------+-----------+-----------------+-------------+----------+
|ts_weekday|ts_hour|revenue   |order_count|prev_hour_revenue|revenue_delta|growth_pct|
+----------+-------+----------+-----------+-----------------+-------------+----------+
|0         |0      |126738.73 |421        |NULL             |NULL         |NULL      |
|0         |1      |186513.04 |720        |126738.73        |59774.31     |47.16     |
|0         |2      |297080.45 |1318       |186513.04        |110567.41    |59.28     |
|0         |3      |723948.74 |2953       |297080.45        |426868.29    |143.69    |
|0         |4      |1103964.61|4578       |723948.74        |380015.87    |52.49     |
|0         |5      |1409847.1 |5808       |1103964.61       |305882.49    |27.71     |
|0         |6      |1648015.09|6776       |1409847.1        |238167.99    |16.89     |
|0         |7      |1844173.48|7270       |1648015.09       |196158.39    |1

In [12]:
print("\n Hoàn thành tất cả 10 câu phân tích.")
spark.stop()


 Hoàn thành tất cả 10 câu phân tích.
